# 01 — Exploratory data analysis

What the competition data looks like before we design anything.

| Part | Data | Question |
|---|---|---|
| A | `train.csv` (58 labelled studies) | How common is each label, and which labels travel together? |
| B | `train.csv` (all 4,407 reports) | What do the reports look like: length, language, sites, and how they relate to the gold labels? |
| C | `train_series.csv` | Which kinds of series does each study have, and how often is a view missing? |

Each part ends with **Findings**. The last cell collects what they mean for later components.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from langdetect import DetectorFactory, LangDetectException, detect

DetectorFactory.seed = 0  # langdetect is random by default; seed it for reproducible results

# Project root = first parent folder that contains pyproject.toml (works wherever the notebook is run from)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = ROOT / "data" / "raw"

LABELS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA",
    "Effusion", "Synovitis", "Baker's",
    "Contusion", "Fracture",
]

train = pd.read_csv(RAW / "train.csv")
series = pd.read_csv(RAW / "train_series.csv")
test_series = pd.read_csv(RAW / "test_series.csv")

assert list(train.columns) == ["StudyInstanceUID", "Report", *LABELS]
train.shape, series.shape

## Part A — Labels

### A1. Which studies are labelled?
Labels are all-or-nothing: a study has either all 12 filled or none.

In [ ]:
n_filled = train[LABELS].notna().sum(axis=1)
print(n_filled.value_counts().sort_index().rename("studies"))

is_labelled = n_filled == len(LABELS)
labelled = train.loc[is_labelled]
assert len(labelled) == 58
assert set(n_filled.unique()) == {0, len(LABELS)}, "some study is only partly labelled"

### A2. Prevalence, with its uncertainty
`se` is the standard error of a proportion, \( \sqrt{p(1-p)/n} \). It assumes a *random* sample, which the 58 are not (see A3), so read it as a lower bound on how uncertain these numbers are.

In [ ]:
n = len(labelled)
prevalence = pd.DataFrame({"n_pos": labelled[LABELS].sum().astype(int)})
prevalence["pct"] = 100 * prevalence["n_pos"] / n
prevalence["se_pct"] = 100 * np.sqrt((prevalence["pct"] / 100) * (1 - prevalence["pct"] / 100) / n)
prevalence.sort_values("n_pos").round(1)

### A3. Positive labels per study

In [ ]:
pos_per_study = labelled[LABELS].sum(axis=1).astype(int)
ax = pos_per_study.value_counts().sort_index().plot.bar(figsize=(6, 3))
ax.set(xlabel="positive labels in the study", ylabel="studies", title="Labelled studies (n=58)")
plt.show()
print("studies with no positive label:", (pos_per_study == 0).sum())

### A4. Which labels occur together?
Phi coefficient = Pearson correlation between two 0/1 columns (−1 … +1).

In [ ]:
phi = labelled[LABELS].corr()
plt.figure(figsize=(8, 6.5))
sns.heatmap(phi, vmin=-1, vmax=1, cmap="RdBu_r", annot=True, fmt=".1f", annot_kws={"size": 7})
plt.title("Phi coefficient between labels (n=58)")
plt.show()

pairs = phi.where(np.triu(np.ones(phi.shape, dtype=bool), k=1)).stack()
pairs.rename("phi").sort_values(key=abs, ascending=False).head(10).round(2)

### Findings — A
- **Only 58 of 4,407 studies (1.3%) are labelled.** The image model cannot be trained on them. Training labels must come from the reports (component 3), and the 58 become the gold set for checking the labeller.
- **The 58 are enriched, not random.** Every one of them has at least one positive, and Fracture is 31%. So their prevalences say nothing about the real (test) rates. AUC is prevalence-free, so the metric itself is not distorted.
- **Rare labels will be hard to evaluate:** MCL has 9 positives, Lateral OA 11 and Baker's 12. Per-label scores on the 58 for these swing a lot from a single study.
- **Two families of labels show up:** *degenerative* (Medial/Lateral OA, PF OA, menisci, Baker's, Synovitis) and *acute trauma* (ACL, Contusion, Fracture). Medial OA and Contusion are negatively correlated (−0.33). With n=58, any single phi below ~0.3 may be noise.

## Part B — Reports

### B1. Length

In [ ]:
words = train["Report"].str.split().str.len()
ax = words.plot.hist(bins=50, figsize=(6, 3))
ax.set(xlabel="words per report", title="Report length (all 4,407)")
plt.show()
words.describe().round(0)

In [ ]:
for label, idx in [("shortest", words.nsmallest(3).index), ("longest (first 400 chars)", words.nlargest(1).index)]:
    print(f"===== {label}")
    for i in idx:
        print(f"[{words[i]} words]", train.at[i, "Report"][:400].replace("\n", " "), "\n")

### B2. Language

In [ ]:
def detect_language(text: str) -> str:
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"


train["lang"] = train["Report"].map(detect_language)
labelled = train.loc[is_labelled]  # refresh so it carries the new column

lang = pd.DataFrame({
    "all_n": train["lang"].value_counts(),
    "labelled_n": labelled["lang"].value_counts(),
}).fillna(0).astype(int)
lang["all_pct"] = (100 * lang["all_n"] / len(train)).round(1)
lang["labelled_pct"] = (100 * lang["labelled_n"] / len(labelled)).round(1)
lang

### B3. Hidden sites (report templates)
Reports from the same hospital share a template. Group reports by their opening characters (lower-cased, digits masked) as a rough site fingerprint.

In [ ]:
opening = (
    train["Report"].str.strip().str.lower().str[:15]
    .str.replace(r"\d", "#", regex=True)
)
template_counts = opening.value_counts()
print("distinct openings:", len(template_counts))
print("openings used by only one report:", (template_counts == 1).sum())
print("share of reports covered by the 20 most common openings:",
      round(template_counts.head(20).sum() / len(train), 2))

top = template_counts.head(15).rename("reports").to_frame()
top["lang"] = [train.loc[opening == t, "lang"].mode()[0] for t in top.index]
top

### B4. Reports next to their gold labels
Read these. Compare what the report says with the gold labels underneath.

In [ ]:
def show_report(row: pd.Series, max_chars: int = 1500) -> None:
    positives = [label for label in LABELS if row[label] == 1]
    print(f"lang={row['lang']}   gold positives: {positives}")
    print(row["Report"][:max_chars])
    print("=" * 100)


for _, row in labelled[labelled["lang"] == "en"].head(3).iterrows():
    show_report(row)

### Findings — B
- **Reports are medium length** (median ~130 words, 8–690). They are free text, sometimes structured by section (ACL / menisci / cartilage …).
- **Some reports carry almost no information:** the shortest say only "Normal" or are empty ("Resultados: . Impresión:"). "Normal" means all 12 = 0, but an empty report means *unknown*. The labeller must tell these apart, and empty ones should be left out of training.
- **Nine languages.** English is only ~39% of reports, followed by Spanish, Turkish, Croatian, Greek, German, Bulgarian, Dutch and French. The labeller must be multilingual.
- **The gold set does not cover every language:** 48% of the 58 are English, and there is **no French report** among them. The labeller's accuracy on the 58 says little about French (and not much about German/Dutch, with 2 each).
- **Sites are only partly visible from the text.** A few big templates dominate (Spanish "Técnica: RMN…", Turkish "SAĞ/SOL DİZ MRG…", Dutch/Belgian "MR Knie…"), but hundreds of reports open with free text. Language is a coarse site proxy; DICOM metadata (scanner model, field strength) may be a better one (checked when we open the images).
- **The gold labels do not follow the reports.** Report 1 says *moderate joint effusion* and *osteochondral fracture*, yet gold has Effusion = 0 and Fracture = 0. Report 3 says *synovitis* with *massive effusion*, yet gold has Synovitis = 0. Report 2 has a *small* effusion and gold Effusion = 0, so there is likely a severity threshold. The gold labels were most likely made by radiologists **looking at the images** with their own definitions, not by reading the reports.
  - **Consequence 1:** even a perfect report reader cannot fully match the gold set. There is a *ceiling*.
  - **Consequence 2:** the test labels are probably made the same way. Report-derived training labels therefore differ *systematically* from what we are scored on. The labeller should learn the annotators' definitions (thresholds such as "small effusion = 0"), not just "is it mentioned?".

## Part C — Series

### C1. Integrity checks

In [ ]:
assert series["SeriesInstanceUID"].is_unique, "duplicate series ids"
assert not series.isna().any().any(), "missing values in train_series.csv"
assert set(series["StudyInstanceUID"]) == set(train["StudyInstanceUID"]), "studies and series do not match"
assert set(series["Anatomical_Plane"]) == {"Sagittal", "Coronal", "Axial"}
print("all integrity checks passed")

### C2. Series per study

In [ ]:
per_study = series.groupby("StudyInstanceUID").size()
ax = per_study.value_counts().sort_index().plot.bar(figsize=(6, 3))
ax.set(xlabel="series in the study", ylabel="studies")
plt.show()
per_study.describe().round(1)

### C3. Which kinds of series exist

In [ ]:
pd.crosstab(series["Anatomical_Plane"], [series["Fluid_Sensitive"], series["Fat_Suppression"]], margins=True)

In [ ]:
flags_agree = (series["Fluid_Sensitive"] == series["Fat_Suppression"]).mean()
test_flags_agree = (test_series["Fluid_Sensitive"] == test_series["Fat_Suppression"]).mean()
print(f"train: flags identical in {flags_agree:.1%} of series")
print(f"test example: flags identical in {test_flags_agree:.1%} of series")

### C4. Coverage: which views each study has
Name each kind of series by plane + sequence type. Since the flags always agree in train, there are only two sequence types:
`FS` = fluid-sensitive with fat suppression (PD/T2 FS, STIR) and `noFS` = no fat suppression.

`noFS` is **not** just T1. The DICOM `SeriesDescription` of the example test study shows PD and even T2 series without fat-sat flagged `0, 0`. In practice the organisers mark only fat-suppressed fluid images as fluid-sensitive.

In [ ]:
series["kind"] = (
    series["Anatomical_Plane"].str[:3].str.lower() + "_"
    + np.where(series["Fluid_Sensitive"] == 1, "FS", "noFS")
)
counts = pd.crosstab(series["StudyInstanceUID"], series["kind"])

coverage = pd.DataFrame({
    "pct_studies_with_one": (100 * (counts > 0).mean()).round(1),
    "pct_studies_with_duplicates": (100 * (counts > 1).mean()).round(1),
})
coverage.sort_values("pct_studies_with_one", ascending=False)

In [ ]:
fs_kinds = ["sag_FS", "cor_FS", "axi_FS"]
has_all_fs = (counts[fs_kinds] > 0).all(axis=1)
print(f"studies with an FS series in all 3 planes: {has_all_fs.mean():.1%}")
print("missing FS view, by plane:")
print((counts[fs_kinds] == 0).sum().rename("studies"))

### Findings — C
- **The data is clean:** every study has series, every series belongs to a study, and there are no duplicate ids or missing values.
- **3–14 series per study, usually 5.**
- **In train, `Fluid_Sensitive` and `Fat_Suppression` are always identical**, so there are only two sequence types per plane: fluid-sensitive fat-sat (PD/T2 FS, STIR) and noFS (no fat-sat: T1, but also PD or T2 without fat-sat). The organisers warn they "are not necessarily equivalent for every case", so the test set may contain combinations never seen in training. Our input logic must not crash on them.
- **The fat-sat fluid views are the backbone.** `axi_FS` is in 100% of studies, `cor_FS` in 96% and `sag_FS` in 94%. 91% have all three. Every study has at least one fat-sat series. noFS is common in sagittal (97%) and coronal (77%), rare in axial (19%).
- **DICOM metadata names the real sequence.** The 86 kept tags include `SeriesDescription`, `RepetitionTime`, `EchoTime`, `Manufacturer`, `ManufacturerModelName` and `MagneticFieldStrength`, so we can derive finer sequence types than the two flags, and a better site proxy than report templates.
- **Duplicates happen:** 7–20% of studies have two series of the same kind (for example two sagittal noFS series). We need a rule for choosing between them, or a model that can take several.

## Summary: what this means for the next components

**Key facts**
1. 58 labelled studies (1.3%). They are enriched and have 9–35 positives per label.
2. The gold labels were likely made from the images, not the reports, and there are visible mismatches.
3. Reports are in 9 languages; the gold set covers 8 (no French) and is 48% English.
4. Site identity is only partly recoverable from report templates.
5. Per plane: a fat-sat fluid series is present in 94–100% of studies; noFS is common except axial.
6. Sequence flags are identical in train but may not be in test.

**Implications**
- **Component 2 (validation):** the 58 are too few and too enriched to be the main validation set on their own. Use CV folds over the 4,349 pseudo-labelled studies *and* report the 58 separately. Group folds by a site proxy (language/template, later DICOM metadata) to catch domain shift.
- **Component 3 (labeller):** multilingual. It must handle negation, uncertainty and severity, and should be tuned to the annotators' definitions using the 58, knowing there is a ceiling. Output probabilities, not just 0/1.
- **Component 4 (inputs):** three core slots `sag_FS`, `cor_FS`, `axi_FS` (+ optional noFS), a rule for duplicates, a fallback for the ~6% of studies missing a sagittal or coronal FS series, and robust handling of unseen flag combinations.